# ArchLens — Research, Visualization & Sensitivity Analysis

Version: 1.00 | Course: AI Agent Orchestration — HW4 (EX04)

A reproducible analysis of ArchLens's token economics, OAT sensitivity, and run-to-run variance, with
the measured artifacts loaded directly from the repository.

## Table of Contents
1. Introduction & Research Questions
2. Methodology (OAT sensitivity design)
3. References
4. Discussion
5. Figures
6. Token Economics
7. ISO/IEC 25010 Quality Mapping
8. Findings & Threats to Validity

In [ ]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "metrics").exists() else Path.cwd().parent


def load(rel):
    return json.loads((ROOT / rel).read_text(encoding="utf-8"))


print("repo root:", ROOT.name)

## 1. Introduction & Research Questions

**RQ1.** How much input-token cost does Graphify-assisted retrieval save versus naive full-context
stuffing? We define token savings as

$$ S = \left(1 - \frac{T_{\text{assisted}}}{T_{\text{baseline}}}\right) \times 100\%. $$

**RQ2.** How sensitive are these outcomes to the key configuration parameters (one-at-a-time design)?

## 2. Methodology (OAT sensitivity design)

We use a One-At-a-Time (OAT) design. Let the baseline configuration be $\mathbf{x}_0$ and let each
swept parameter $p_i \in \{\text{analysis\_depth},\ \text{top\_k\_pages},\ \text{rate\_limit\_rpm},\
\text{similarity\_threshold}\}$ range over $V_i$. A variant holds every other parameter at baseline:

$$ \mathbf{x}_{i,v} = \mathbf{x}_0 \oplus (p_i = v), \qquad v \in V_i. $$

The one-factor effect of $p_i$ at value $v$ on an outcome $f$ is the delta

$$ \delta_{i}(v) = f(\mathbf{x}_{i,v}) - f(\mathbf{x}_0). $$

Each baseline configuration is run $N \ge 3$ times to quantify run-to-run variance.

## 3. References

1. Liu, N. F., Lin, K., Hewitt, J., et al. (2024). *Lost in the Middle: How Language Models Use Long
   Contexts.* Transactions of the ACL (TACL).
2. Vaswani, A., Shazeer, N., Parmar, N., et al. (2017). *Attention Is All You Need.* NeurIPS.
3. Newman, M. E. J. (2006). *Modularity and community structure in networks.* PNAS, 103(23).
4. Lewis, P., Perez, E., Piktus, A., et al. (2020). *Retrieval-Augmented Generation for
   Knowledge-Intensive NLP Tasks.* NeurIPS.

## 4. Discussion

Liu et al. (2024) show that language models use information at the **start** of their context far
more reliably than information buried in the middle of a long prompt. ArchLens exploits this directly:
instead of stuffing the entire source tree into the context (where most of it lands in the
low-attention middle), it retrieves a graph-scoped slice — the `index.md` hub read first, then 2-3
wiki pages — so the highest-value evidence sits at the front of the prompt. The `hot.md`-first reading
order is a deliberate application of the same finding. The token-savings result (Figure 1) quantifies
the cost side of this design.

## 5. Figures

![Figure 1: Tokens per question — naive baseline vs Graphify-assisted.](../docs/assets/tokens_bar.png)
*Figure 1. Baseline vs assisted tokens per question (live gpt-4.1-mini).*

![Figure 2: Token savings decomposed by pipeline stage.](../docs/assets/tokens_waterfall.png)
*Figure 2. Waterfall of token savings by stage; the final bar equals the total savings.*

![Figure 3: Similarity threshold vs validated duplicates.](../docs/assets/similarity_scatter.png)
*Figure 3. Scatter with the 0.91 triage threshold annotated.*

![Figure 4: OAT sensitivity magnitude.](../docs/assets/sensitivity_heatmap.png)
*Figure 4. Heatmap of normalized outcome delta per swept parameter.*

![Figure 5: Run-to-run variance.](../docs/assets/variance_box.png)
*Figure 5. Box plot of tokens and runtime across the 3 baseline runs.*

In [ ]:
metrics = load("metrics/out/token_metrics.json")
savings = metrics["savings"]
print(f"Token savings: {savings['savings_pct']:.2f}%  (target met: {metrics['target_met']})")
print(f"Break-even queries: {metrics['amortization']['break_even_queries']}")
pd.DataFrame(metrics["per_model"])

## 6. Token Economics

The per-model cost table above lists input/output tokens and USD cost. Measured savings vs the naive
baseline exceed the 70% target (see Figure 1), and the one-time Graphify build cost is amortized
within a handful of queries (Figure 2).

In [ ]:
from IPython.display import Markdown

from archlens.metrics.iso25010_mapping import iso25010_table

Markdown(iso25010_table({
    "coverage_pct": 97, "modularity": 144,
    "token_savings_pct": round(savings["savings_pct"], 1), "ruff_violations": 0,
    "tests_green": True, "spof_count": 1, "uv_only": True, "usability_resolved": True}))

## 8. Findings & Threats to Validity

**Findings.**
1. Graphify-assisted retrieval cuts input tokens by well over the 70% target (Figure 1, Figure 2).
2. Rate-limit throughput is the most sensitive OAT parameter for wait time (Figure 4).

**Threats to validity.** The token-economics and run-to-run-variance figures are real live
measurements on gpt-4.1-mini; per-question token counts are near-deterministic, so the measured
variance is dominated by wall-clock runtime (Figure 5). The similarity-threshold sweep is bounded by
the analyzed graph's confidence distribution. Results are reported for a single target repository
(httpie).